# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedr456/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

My lane's question is **"which pages first?"** — a ranking problem, not a yes/no. So the
method is a **classifier's probability used as a ranking score, judged at Precision@50**
(the queue an editor can actually work through), exactly like the frozen Week-4 baseline.

Two models, in honesty order: **Logistic Regression first** (readable; if it wins, we're
done cheaply), then **Random Forest** (interactions — which §Week-4 already proved matter:
staleness only works *paired* with visibility). Floors reported next to everything: the
base rate and a 50-shuffle random queue.

Features are the safe past-window columns from the data contract, with the two
missingness-pattern flags the data dictionary demands (`avg_position = 0` means "no data",
word_count gaps follow content type — blind fills would inject a category signal).

In [1]:
# Setup + the feature frame (safe columns only, missingness handled with flags).
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mohamedr456/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"].str.lower() == "down").astype(int)

X = pd.DataFrame({
    "content_age_days": df["content_age_days"],
    "days_since_last_update": df["days_since_last_update"],
    "log_impressions_90d": np.log1p(df["impressions_90d"]),
    "avg_position": df["avg_position"].replace(0, np.nan),   # 0 = no data, not rank 0
    "has_position": (df["avg_position"] > 0).astype(int),
    "ctr": df["ctr"],
    "word_count": df["word_count"],
    "has_word_count": df["word_count"].notna().astype(int),
    "engagement_rate": df["engagement_rate"],
})
X["avg_position"] = X["avg_position"].fillna(X["avg_position"].median())
X["word_count"] = X["word_count"].fillna(X["word_count"].median())
X["engagement_rate"] = X["engagement_rate"].fillna(0)

FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
assert not set(X.columns) & FORBIDDEN, "leak: forbidden column in features"
print(f"{len(X):,} rows, {X.shape[1]} features, 0 NaNs left: {X.isna().sum().sum()==0}")
print("features:", list(X.columns))

30,000 rows, 9 features, 0 NaNs left: True
features: ['content_age_days', 'days_since_last_update', 'log_impressions_90d', 'avg_position', 'has_position', 'ctr', 'word_count', 'has_word_count', 'engagement_rate']


## 2. Split design

**GroupKFold(5) grouped by `client_id` — client-holdout.** Rows within one client share
hidden character (vertical, audience, template habits); a random split would let the model
memorize the client and fake skill. The deployment question is *"does the queue work for a
client the model has never seen?"* — so every fold's test clients are fully unseen, and
every number below is out-of-fold.

Named honestly: a **time split isn't possible on this release** — the starter snapshot is
one trailing-90-day window with no history axis. Time-aware validation is exactly what the
warehouse panel adds later (ML-09 revisits this); within this snapshot, client-holdout is
the strongest honest design available.

In [2]:
# The split, proved: no client appears on both sides of any fold.
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
overlaps = 0
for tr, te in gkf.split(X, y, groups=df["client_id"]):
    overlaps += len(set(df["client_id"].iloc[tr]) & set(df["client_id"].iloc[te]))
print("client overlap across all folds:", overlaps, "-> client-holdout holds")
print("clients per fold (test):",
      [df['client_id'].iloc[te].nunique() for _, te in gkf.split(X, y, groups=df['client_id'])])

client overlap across all folds: 0 -> client-holdout holds
clients per fold (test): [1, 7, 8, 8, 8]


## 3. Train + compare vs my baseline

Same 30,000 rows, same label, same Precision@50, same folds for every line in the table —
the frozen Week-4 rule is scored on each fold's test clients too. The headline (mean of 5
client-holdout folds):

| queue | mean P@50 |
|---|---|
| random order (50 shuffles/fold) | **0.544** (= base rate) |
| frozen Week-4 rule | **0.560** |
| logistic regression | **0.512** |
| random forest | **0.700** |

Two honest findings inside that table. **The negative result:** logistic regression loses
to a random queue on 2 of 5 folds and on the mean — the signals are non-linear and
interactive; a linear scorer genuinely can't use them. **The win:** the random forest beats
the frozen rule by **+0.14 absolute** (0.560 → 0.700) *at real depth 50*, where the rule
starves (35 scorable pages). Fold spread is wide (0.56–0.78) — small client sets differ;
reported, not hidden.

In [3]:
# The comparison table, computed out-of-fold, plus committed receipts.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

def p_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

rule_score = ((df["days_since_last_update"] >= 180) &
              (df["impressions_90d"] >= 100)).astype(int) * df["impressions_90d"]

rng = np.random.default_rng(0)
rows, oof_rf = [], np.zeros(len(X))
for fold, (tr, te) in enumerate(gkf.split(X, y, groups=df["client_id"])):
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                n_jobs=-1, random_state=42).fit(X.iloc[tr], y.iloc[tr])
    lr = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=2000)).fit(X.iloc[tr], y.iloc[tr])
    p_rf = rf.predict_proba(X.iloc[te])[:, 1]; oof_rf[te] = p_rf
    p_lr = lr.predict_proba(X.iloc[te])[:, 1]
    yte = y.iloc[te]
    rows.append({"fold": fold, "n_test": len(te), "base_rate": round(yte.mean(), 3),
                 "p50_random": round(np.mean([p_at_k(rng.random(len(te)), yte) for _ in range(50)]), 3),
                 "p50_rule": round(p_at_k(rule_score.iloc[te], yte), 3),
                 "p50_logreg": round(p_at_k(p_lr, yte), 3),
                 "p50_rf": round(p_at_k(p_rf, yte), 3),
                 "auc_rf": round(roc_auc_score(yte, p_rf), 3)})
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print("\nmeans:")
print(res.drop(columns=["fold", "n_test"]).mean().round(3).to_string())

import json as _json
os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/model_metrics.json", "w") as f:
    _json.dump({"design": "GroupKFold(5) by client_id, all numbers out-of-fold",
                "metric": "precision_at_50 (+ auc_rf)", "seed": 42,
                "per_fold": rows,
                "means": res.drop(columns=["fold", "n_test"]).mean().round(3).to_dict()},
               f, indent=2)
print("\nreceipts committed: work/outputs/model_metrics.json")

 fold  n_test  base_rate  p50_random  p50_rule  p50_logreg  p50_rf  auc_rf
    0    7008      0.490       0.503      0.38        0.48    0.78   0.648
    1    5731      0.645       0.646      0.60        0.40    0.72   0.579
    2    5753      0.379       0.358      0.38        0.20    0.56   0.720
    3    5755      0.622       0.646      0.70        0.66    0.72   0.671
    4    5753      0.585       0.566      0.74        0.82    0.72   0.646

means:
base_rate     0.544
p50_random    0.544
p50_rule      0.560
p50_logreg    0.512
p50_rf        0.700
auc_rf        0.653

receipts committed: work/outputs/model_metrics.json


## 4. Errors and interpretation

**What the model leans on** (importances below): visibility (`log_impressions_90d`),
`avg_position`, and `content_age_days` carry it; `days_since_last_update` — the *core* of
the hand rule — ranks near last (≈0.05). That is the Week-4 MIXED verdict, relearned by the
model on its own: staleness barely matters alone; reach and position interactions do the
work. Nothing suspiciously perfect — the top feature explains ~0.27, not 0.9 (a 0.9 would
mean leakage, per the attack checklist).

**Where it's wrong:** the three hardest false positives below are highly-visible,
mid-position pages the model ranks near the top that did *not* decline this window —
exactly the "big and probably fine" pages an editor would also over-flag. The model is
directional decision-support: it orders a review queue, it does not diagnose a page.

**Fold spread** (P@50 0.56–0.78) tracks each fold's base rate (0.38–0.65): with ~10 test
clients per fold, client mix moves every number. The warehouse-scale rerun is what
tightens this — named in limitations, not hidden.

In [4]:
# Importances + three concrete wrong cases from the out-of-fold ranking.
rf_full = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                 n_jobs=-1, random_state=42).fit(X, y)
imp = pd.Series(rf_full.feature_importances_, index=X.columns).sort_values(ascending=False)
print("RF feature importances (fit on all rows, for reading only):")
print(imp.round(3).to_string())

order = np.argsort(-oof_rf)
top50 = order[:50]
fp = [i for i in top50 if y.iloc[i] == 0][:3]
cols = ["content_id", "content_type", "content_age_days", "days_since_last_update",
        "impressions_90d", "avg_position", "ctr"]
print("\n3 hardest false positives (top-50 by out-of-fold score, not declining):")
print(df.iloc[fp][cols].assign(oof_score=np.round(oof_rf[fp], 3)).to_string(index=False))
print("\nWhy they're hard: big reach + mid position looks exactly like the declining")
print("profile; without page content or history depth, reach-heavy stable pages are")
print("indistinguishable from reach-heavy fading ones in a single 90-day snapshot.")

RF feature importances (fit on all rows, for reading only):
log_impressions_90d       0.268
avg_position              0.184
content_age_days          0.173
word_count                0.130
ctr                       0.098
engagement_rate           0.048
days_since_last_update    0.045
has_position              0.040
has_word_count            0.015

3 hardest false positives (top-50 by out-of-fold score, not declining):
          content_id    content_type  content_age_days  days_since_last_update  impressions_90d  avg_position  ctr  oof_score
content_78a8704f334f keyword article               154                      20             3818          20.0  0.1      0.978
content_9bf53d7bbea5  feedly article                98                      20              334          29.6  0.0      0.968
content_af812e811d6f keyword article               165                      20              284          21.3  0.0      0.962

Why they're hard: big reach + mid position looks exactly like the declinin

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.